# **TRAINING MODEL**

In [ ]:
import os
import csv
from datetime import datetime
import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import case_config as cfg


TRAIN_SIZE = 0.8
# ============================================================
# TODO: FILES CONFIGURATION 
# ============================================================

TRAIN_FILE = r"train.csv"
MODEL_FILE = r"model.joblib" # model output 

TRAIN_SIZE = 0.8
LOG_FILE = r"team_trainig_submit.csv"

# ============================================================
# NOTE: HYPERPARAMETERS TUING
# ============================================================
# Recommended tuning order:
#   1. max_depth
#   2. min_samples_leaf
#   3. min_samples_split
#   4. random_state
#   5. criterion

CRITERION = "gini"          # Try: "gini", "entropy"
MAX_DEPTH = None          # Try: 4, 6, 8, 10, None
MIN_SAMPLES_SPLIT = 2       # Try: 2, 5, 10, 20
MIN_SAMPLES_LEAF = 1       # Try: 1, 2, 5, 10
RANDOM_STATE = 42

# ============================================================
# LOAD DATA
# ============================================================
def load_labelled(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=cfg.FEATURES + ["y"])
    x = df[cfg.FEATURES].to_numpy(dtype=float)
    y = df["y"].astype(int).to_numpy()
    return df, x, y

def show_class_balance(df, title):
    print("")
    print("===== " + title + " =====")
    counts = df["y"].value_counts().sort_index()
    for class_id, n in counts.items():
        print(
            "  %d %-10s %5d  %5.1f%%"
            % (
                class_id,
                cfg.label_of(class_id),
                n,
                100.0 * n / len(df)
            )
        )

# ============================================================
# BUILD MODEL
# ============================================================

def build_model():
    return DecisionTreeClassifier(
        criterion=CRITERION,
        max_depth=MAX_DEPTH,
        min_samples_split=MIN_SAMPLES_SPLIT,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        random_state= RANDOM_STATE,
    )
    
# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, x, y, title):
    predicted = model.predict(x)
    score = f1_score(
        y,
        predicted,
        average="macro",
        labels=cfg.class_ids()
    )
    print("")
    print("===== " + title + " =====")
    print("Macro-F1: %.4f" % score)
    print(
        classification_report(
            y,
            predicted,
            labels=cfg.class_ids(),
            target_names=cfg.CLASS_LABELS,
            digits=4,
            zero_division=0
        )
    )
    print("Confusion matrix (rows = true, columns = predicted):")
    print(
        confusion_matrix(
            y,
            predicted,
            labels=cfg.class_ids()
        )
    )
    return score

def save_log(timestamp, criterion, max_depth, min_samples_split, min_samples_leaf, random_state, val_f1):
    file_exists = os.path.isfile(LOG_FILE)
    
    headers = [
        "timestamp", 
        "criterion", 
        "max_depth", 
        "min_samples_split", 
        "min_samples_leaf",
        "random_state", 
        "val_macro_f1"
    ]
    
    data = [
        timestamp, 
        criterion, 
        max_depth, 
        min_samples_split, 
        min_samples_leaf,
        random_state, 
        round(val_f1, 4)
    ]
    
    with open(LOG_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
        writer.writerow(data)
def main():
    print("Case:  " + cfg.CASE_TITLE)
    print("Model: " + cfg.MODEL_NAME)

    # NOTE: Load Train.csv
    train_df, x, y = load_labelled(TRAIN_FILE)
    print("")
    print("Total rows in train.csv: %d" % len(train_df))
    show_class_balance(train_df, "CLASS BALANCE: TRAINING")

    # NOTE: Split train.csv into TRAIN_SIZE train / validation
    x_train, x_val, y_train, y_val = train_test_split(x, y, train_size=TRAIN_SIZE, stratify=y)

    print("Training rows:   %d" % len(x_train))
    print("Validation rows: %d" % len(x_val))

    # NOTE: Train model using the training part
    model = build_model()
    model.fit(x_train, y_train)
    print("Model trained on %d%% training subset." % round(TRAIN_SIZE * 100))
    print(model)

    # NOTE: Evaluate using the validation part
    validation_f1 = evaluate(model, x_val, y_val, "VALIDATION SET")

    # NOTE: Final model using ALL train.csv
    final_model = build_model()
    final_model.fit(x,y)
    print("===== FINAL MODEL TRAINING =====")
    print("Final model trained on all %d rows in train.csv." % len(x))

    # NOTE: Save Final model
    joblib.dump(final_model, MODEL_FILE)
    print("Saved model: " + MODEL_FILE)
    print("Validation Macro-F1: %.4f" % validation_f1)

    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    save_log(
        timestamp=current_time,
        criterion=CRITERION,
        max_depth=MAX_DEPTH,
        min_samples_split=MIN_SAMPLES_SPLIT,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        random_state= RANDOM_STATE,
        val_f1=validation_f1
    )
    print("Log saved to: " + LOG_FILE)
if __name__ == "__main__":
    main()

# **TESTING MODEL**

In [ ]:
import os
import csv
from datetime import datetime
import joblib
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import case_config as cfg

# ============================================================
# TODO: FILES CONFIGURATION 
# ============================================================
TEST_FILE = r"test.csv"
MODEL_FILE = r"model.joblib" # same file cell 1 saves
LOG_FILE = r"team_testing_submit.csv"  

# ============================================================
# LOAD DATA
# ============================================================

def load_labelled(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=cfg.FEATURES + ["y"])
    x = df[cfg.FEATURES].to_numpy(dtype=float)
    y = df["y"].astype(int).to_numpy()
    return df, x, y

def show_class_balance(df):
    print("")
    print("===== CLASS BALANCE: UNSEEN TEST SET =====")
    counts = df["y"].value_counts().sort_index()
    for class_id, n in counts.items():
        print(
            "  %d %-10s %5d  %5.1f%%"
            % (
                class_id,
                cfg.label_of(class_id),
                n,
                100.0 * n / len(df)
            )
        )

# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, x, y):
    predicted = model.predict(x)
    score = f1_score(
        y,
        predicted,
        average="macro",
        labels=cfg.class_ids()
    )
    print("")
    print("===== FINAL UNSEEN TEST RESULT =====")
    print("Macro-F1: %.4f" % score)
    print(
        classification_report(
            y,
            predicted,
            labels=cfg.class_ids(),
            target_names=cfg.CLASS_LABELS,
            digits=4,
            zero_division=0
        )
    )
    print("Confusion matrix (rows = true, columns = predicted):")
    print(
        confusion_matrix(
            y,
            predicted,
            labels=cfg.class_ids()
        )
    )
    return score
# ============================================================
# FUNCTION FOR LOGGING TEST RESULTS
# ============================================================
def save_test_log(timestamp, model_file, params, test_f1):
    file_exists = os.path.isfile(LOG_FILE)
    
    headers = [
        "timestamp", 
        "model_file",
        "criterion", 
        "max_depth", 
        "min_samples_split", 
        "min_samples_leaf",
        "random_state", 
        "test_macro_f1"
    ]
    
    data = [
        timestamp,
        model_file,
        params.get("criterion"),
        params.get("max_depth"),
        params.get("min_samples_split"),
        params.get("min_samples_leaf"),
        params.get("random_state"),
        round(test_f1, 4)
    ]
    
    with open(LOG_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
        writer.writerow(data)
def main():
    print("Case:  " + cfg.CASE_TITLE)
    print("Model: " + cfg.MODEL_NAME)

    # NOTE: Load trained model
    model = joblib.load(MODEL_FILE)
    print("Loaded model: " + MODEL_FILE)

    # NOTE: Load test dataset
    test_df, x_test, y_test = load_labelled(TEST_FILE)
    print("Unseen test rows: %d" % len(test_df))
    show_class_balance(test_df)

    # NOTE: Final evaluation
    final_f1 = evaluate(
        model,
        x_test,
        y_test
    )
    print("Final unseen-test Macro-F1: %.4f" % final_f1)
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    params = model.get_params() 
    
    save_test_log(
        timestamp=current_time,
        model_file=MODEL_FILE,
        params=params,
        test_f1=final_f1
    )
    print("Test Log saved to: " + LOG_FILE)

if __name__ == "__main__":
    main()